# Serving Metrics and Prefill–Decode Disaggregation

The full code of [Serving Metrics and Prefill–Decode Disaggregation](https://residual-stream.lvtan71.workers.dev/posts/serving-metrics-and-disaggregation/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/serving-metrics-and-disaggregation) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/serving-metrics-and-disaggregation.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "serving-metrics-and-disaggregation"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `serving_sim.py`

A step-by-step simulation of serving Llama-3-8B shapes on H100s, each step's time from the roofline: Poisson arrivals, continuous batching with whole or chunked prefill on colocated GPUs, and prefill and decode on separate GPUs with the KV cache sent over a link. Prints TTFT, TPOT and the worst gap between tokens, and the share of requests within two SLOs, on 2 and 4 GPUs.

In [ ]:
!{sys.executable} serving_sim.py

The post printed:

```text
KV cache per token 128 KiB; a 2,000-token prompt's cache is 250 MiB, 5.2 ms over a 50 GB/s link; its prefill alone takes 65 ms; a step stays weight-bound up to 148 tokens

2 GPUs; share of requests whose first token comes within 2 s and every later token within 50 ms (loose) or 20 ms (tight)
req/s design                     TTFT p50    p99  TPOT p99 worst gap p99  loose  tight
    4 colocated, whole prefill     0.07s   0.47s    10.8ms    517.8ms    16%     5%
    4 colocated, chunks of 512     0.07s   0.48s     8.7ms     16.9ms   100%   100%
    4 colocated, chunks of 128     0.09s   0.59s     5.8ms      6.1ms   100%   100%
    4 disaggregated 1 + 1          0.09s   0.61s     6.6ms     44.8ms   100%    83%
    8 colocated, whole prefill     0.08s   0.52s    21.3ms    517.9ms     2%     0%
    8 colocated, chunks of 512     0.09s   0.54s    12.4ms     17.4ms   100%   100%
    8 colocated, chunks of 128     0.13s   1.04s     6.2ms      6.6ms   100%   100%
    8 disaggregated 1 + 1          0.22s   2.11s     7.2ms     43.8ms    99%    77%
   12 colocated, whole prefill     0.10s   0.60s    27.2ms    518.8ms     0%     0%
   12 colocated, chunks of 512     0.10s   0.65s    14.5ms     17.5ms   100%   100%
   12 colocated, chunks of 128     0.21s   1.45s     6.2ms      6.7ms   100%   100%
   12 disaggregated 1 + 1          6.25s  10.05s     7.3ms     43.3ms     6%     4%

4 GPUs
req/s design                     TTFT p50    p99  TPOT p99 worst gap p99  loose  tight
   24 colocated, chunks of 128     0.20s   1.31s     6.2ms      6.7ms   100%   100%
   24 disaggregated 2 + 2          5.85s  12.80s     7.9ms     45.7ms    12%     9%
   24 disaggregated 3 + 1          0.10s   0.51s    18.9ms     57.7ms    98%     7%
```